Pipeline automatisé avec Streams & Tasks pour détecter et transformer les nouvelles données.
*Co-authored with CoCo*

## 1. Streams — Détection automatique des changements (CDC)

Un **Stream** est un objet Snowflake qui capture les changements (INSERT, UPDATE, DELETE) sur une table depuis la dernière consommation. C'est le mécanisme de **Change Data Capture (CDC)** natif de Snowflake.

**Pourquoi ?**
- On ne veut pas re-transformer 100% des données à chaque exécution
- Le Stream ne stocke pas les données, il pointe vers un offset dans l'historique de la table
- Quand une Task consomme le Stream (via DML), l'offset avance automatiquement
- `SYSTEM$STREAM_HAS_DATA()` permet de ne déclencher la Task que s'il y a de nouvelles données

On crée un Stream sur chaque table RAW qu'on veut surveiller.

In [ ]:
%%sql -r res_change_tracking
-- Activer le change tracking sur les tables RAW (requis pour les streams)
ALTER TABLE SHOPFLOW_DB.RAW.RAW_ORDERS SET CHANGE_TRACKING = TRUE;
ALTER TABLE SHOPFLOW_DB.RAW.RAW_ORDER_ITEMS SET CHANGE_TRACKING = TRUE;
ALTER TABLE SHOPFLOW_DB.RAW.WEB_EVENTS_RAW SET CHANGE_TRACKING = TRUE;
ALTER TABLE SHOPFLOW_DB.RAW.PRODUCTS_RAW SET CHANGE_TRACKING = TRUE;
ALTER TABLE SHOPFLOW_DB.RAW.RAW_CUSTOMERS SET CHANGE_TRACKING = TRUE;

In [ ]:
%%sql -r res_streams
CREATE STREAM IF NOT EXISTS SHOPFLOW_DB.RAW.STR_ORDERS
  ON TABLE SHOPFLOW_DB.RAW.RAW_ORDERS
  APPEND_ONLY = TRUE
  COMMENT = 'Capture les nouvelles commandes insérées dans RAW_ORDERS';

CREATE STREAM IF NOT EXISTS SHOPFLOW_DB.RAW.STR_ORDER_ITEMS
  ON TABLE SHOPFLOW_DB.RAW.RAW_ORDER_ITEMS
  APPEND_ONLY = TRUE
  COMMENT = 'Capture les nouveaux items insérés dans RAW_ORDER_ITEMS';

CREATE STREAM IF NOT EXISTS SHOPFLOW_DB.RAW.STR_WEB_EVENTS
  ON TABLE SHOPFLOW_DB.RAW.WEB_EVENTS_RAW
  APPEND_ONLY = TRUE
  COMMENT = 'Capture les nouveaux événements web insérés dans WEB_EVENTS_RAW';

CREATE STREAM IF NOT EXISTS SHOPFLOW_DB.RAW.STR_PRODUCTS
  ON TABLE SHOPFLOW_DB.RAW.PRODUCTS_RAW
  APPEND_ONLY = TRUE
  COMMENT = 'Capture les nouveaux produits insérés dans PRODUCTS_RAW';

CREATE STREAM IF NOT EXISTS SHOPFLOW_DB.RAW.STR_CUSTOMERS
  ON TABLE SHOPFLOW_DB.RAW.RAW_CUSTOMERS
  APPEND_ONLY = TRUE
  COMMENT = 'Capture les nouveaux clients insérés dans RAW_CUSTOMERS';

## 2. Tables cibles dans STAGING

**Pourquoi un schéma STAGING séparé ?**
- Les données RAW sont brutes (tout en VARCHAR, pas de typage, potentiels nulls)
- STAGING contient les données **nettoyées, typées et normalisées** :
  - VARCHAR → DATE, TIMESTAMP, NUMBER (typage fort)
  - Gestion des NULLs et valeurs vides
  - Parsing des champs VARIANT (JSON → colonnes plates)
  - Ajout de métadonnées de traçabilité (`_LOADED_AT`)

Ces tables STAGING serviront de source pour les modèles analytiques dans MARTS.

In [ ]:
%%sql -r res_stg_orders
CREATE TABLE IF NOT EXISTS SHOPFLOW_DB.STAGING.STG_ORDERS (
  ORDER_ID        VARCHAR NOT NULL,
  CUSTOMER_ID     VARCHAR NOT NULL,
  ORDER_DATE      DATE,
  STATUS          VARCHAR,
  TOTAL_AMOUNT    NUMBER(12,2),
  _LOADED_AT      TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
)
COMMENT = 'Commandes nettoyées et typées';

In [ ]:
%%sql -r res_stg_items
CREATE TABLE IF NOT EXISTS SHOPFLOW_DB.STAGING.STG_ORDER_ITEMS (
  ORDER_ITEM_ID   VARCHAR NOT NULL,
  ORDER_ID        VARCHAR NOT NULL,
  PRODUCT_ID      VARCHAR NOT NULL,
  QUANTITY        INT,
  UNIT_PRICE      NUMBER(10,2),
  LINE_TOTAL      NUMBER(12,2),
  _LOADED_AT      TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
)
COMMENT = 'Lignes de commande nettoyées avec calcul LINE_TOTAL';

In [ ]:
%%sql -r res_stg_events
CREATE TABLE IF NOT EXISTS SHOPFLOW_DB.STAGING.STG_WEB_EVENTS (
  EVENT_ID        VARCHAR NOT NULL,
  EVENT_TYPE      VARCHAR,
  USER_ID         VARCHAR,
  SESSION_ID      VARCHAR,
  EVENT_TS        TIMESTAMP_NTZ,
  DEVICE          VARCHAR,
  PRODUCT_ID      VARCHAR,
  IP_ADDRESS      VARCHAR,
  REFERRER        VARCHAR,
  USER_AGENT      VARCHAR,
  _LOADED_AT      TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
)
COMMENT = 'Événements web avec VARIANT parsé en colonnes plates';

In [ ]:
%%sql -r res_stg_products
CREATE TABLE IF NOT EXISTS SHOPFLOW_DB.STAGING.STG_PRODUCTS (
  PRODUCT_ID      VARCHAR NOT NULL,
  NAME            VARCHAR,
  BRAND           VARCHAR,
  CATEGORY        VARCHAR,
  PRICE           NUMBER(10,2),
  COLOR           VARCHAR,
  WARRANTY_MONTHS INT,
  WEIGHT_G        INT,
  TAGS            ARRAY,
  _LOADED_AT      TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
)
COMMENT = 'Produits avec attributs JSON aplatis';

In [ ]:
%%sql -r res_stg_customers
CREATE TABLE IF NOT EXISTS SHOPFLOW_DB.STAGING.STG_CUSTOMERS (
  CUSTOMER_ID     VARCHAR NOT NULL,
  FIRST_NAME      VARCHAR,
  LAST_NAME       VARCHAR,
  FULL_NAME       VARCHAR,
  EMAIL           VARCHAR,
  CITY            VARCHAR,
  COUNTRY         VARCHAR,
  CREATED_AT      TIMESTAMP_NTZ,
  _LOADED_AT      TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
)
COMMENT = 'Clients nettoyés avec nom complet calculé';

## 3. Requêtes de transformation

Testons les requêtes de transformation avant de les encapsuler dans des Tasks.

Chaque transformation applique :
- **Typage** : CAST des VARCHAR vers les types appropriés (DATE, NUMBER, TIMESTAMP)
- **Nettoyage** : TRY_CAST pour gérer les valeurs non convertibles sans erreur
- **Enrichissement** : calcul de colonnes dérivées (LINE_TOTAL, FULL_NAME)
- **Parsing VARIANT** : extraction des champs JSON imbriqués avec la notation `:`

In [ ]:
%%sql -r res_test_orders
-- Test transformation ORDERS (depuis le stream)
SELECT
  ORDER_ID,
  CUSTOMER_ID,
  TRY_TO_DATE(ORDER_DATE) AS ORDER_DATE,
  UPPER(TRIM(STATUS)) AS STATUS,
  TRY_TO_NUMBER(TOTAL_AMOUNT, 12, 2) AS TOTAL_AMOUNT
FROM SHOPFLOW_DB.RAW.STR_ORDERS
LIMIT 10;

In [ ]:
%%sql -r res_test_items
-- Test transformation ORDER_ITEMS (depuis le stream)
SELECT
  ORDER_ITEM_ID,
  ORDER_ID,
  PRODUCT_ID,
  TRY_TO_NUMBER(QUANTITY, 10, 0) AS QUANTITY,
  TRY_TO_NUMBER(UNIT_PRICE, 10, 2) AS UNIT_PRICE,
  TRY_TO_NUMBER(QUANTITY, 10, 0) * TRY_TO_NUMBER(UNIT_PRICE, 10, 2) AS LINE_TOTAL
FROM SHOPFLOW_DB.RAW.STR_ORDER_ITEMS
LIMIT 10;

In [ ]:
%%sql -r res_test_events
-- Test transformation WEB_EVENTS (parsing VARIANT CONTEXT)
SELECT
  e.EVENT_ID,
  e.EVENT_TYPE,
  e.USER_ID,
  e.SESSION_ID,
  e.EVENT_TS,
  ext.DEVICE,
  ext.PRODUCT_ID,
  ext.CONTEXT:ip::VARCHAR AS IP_ADDRESS,
  ext.CONTEXT:referrer::VARCHAR AS REFERRER,
  ext.CONTEXT:user_agent::VARCHAR AS USER_AGENT
FROM SHOPFLOW_DB.RAW.STR_WEB_EVENTS e
JOIN SHOPFLOW_DB.RAW.WEB_EVENTS_EXT ext ON e.EVENT_ID = ext.EVENT_ID
LIMIT 10;

In [ ]:
%%sql -r res_test_products
-- Test transformation PRODUCTS (parsing VARIANT DATA)
SELECT
  DATA:product_id::VARCHAR AS PRODUCT_ID,
  DATA:name::VARCHAR AS NAME,
  DATA:brand::VARCHAR AS BRAND,
  DATA:category::VARCHAR AS CATEGORY,
  DATA:price::NUMBER(10,2) AS PRICE,
  DATA:attributes.color::VARCHAR AS COLOR,
  DATA:attributes.warranty_months::INT AS WARRANTY_MONTHS,
  DATA:attributes.weight_g::INT AS WEIGHT_G,
  DATA:tags AS TAGS
FROM SHOPFLOW_DB.RAW.STR_PRODUCTS
LIMIT 10;

In [ ]:
%%sql -r res_test_customers
-- Test transformation CUSTOMERS
SELECT
  CUSTOMER_ID,
  INITCAP(TRIM(FIRST_NAME)) AS FIRST_NAME,
  INITCAP(TRIM(LAST_NAME)) AS LAST_NAME,
  INITCAP(TRIM(FIRST_NAME)) || ' ' || INITCAP(TRIM(LAST_NAME)) AS FULL_NAME,
  LOWER(TRIM(EMAIL)) AS EMAIL,
  TRIM(CITY) AS CITY,
  UPPER(TRIM(COUNTRY)) AS COUNTRY,
  TRY_TO_TIMESTAMP_NTZ(CREATED_AT) AS CREATED_AT
FROM SHOPFLOW_DB.RAW.STR_CUSTOMERS
LIMIT 10;

## 4. Création des Tasks planifiées

**Qu'est-ce qu'une Task ?**
Une Task est un job planifié dans Snowflake qui exécute du SQL selon un schedule (CRON ou intervalle). Combinée à un Stream, elle forme un pipeline ELT réactif :

1. De nouvelles données arrivent dans RAW (via COPY INTO)
2. Le Stream détecte les changements
3. La Task se déclenche (toutes les 5 min) et vérifie `SYSTEM$STREAM_HAS_DATA()`
4. Si le Stream a des données, la Task exécute le INSERT INTO ... SELECT transformé
5. Le Stream avance son offset → prêt pour le prochain batch

**Architecture des Tasks (chaînage avec AFTER) :**
```
TSK_ROOT_SCHEDULER (root, toutes les 5 min)
  ├── TSK_LOAD_STG_ORDERS (AFTER root)
  ├── TSK_LOAD_STG_ORDER_ITEMS (AFTER root)
  ├── TSK_LOAD_STG_WEB_EVENTS (AFTER root)
  ├── TSK_LOAD_STG_PRODUCTS (AFTER root)
  └── TSK_LOAD_STG_CUSTOMERS (AFTER root)
```

La task root ne fait rien, elle sert juste de déclencheur. Les tasks enfants s'exécutent en parallèle après elle.

In [ ]:
%%sql -r res_tsk_root
-- Task root : déclencheur du DAG (toutes les 5 minutes)
CREATE OR REPLACE TASK SHOPFLOW_DB.RAW.TSK_ROOT_SCHEDULER
  WAREHOUSE = WH_TRANSFORM
  SCHEDULE = '5 MINUTE'
  COMMENT = 'Task root qui déclenche le DAG de transformation'
AS
  SELECT 1;  -- No-op, sert uniquement de déclencheur

In [ ]:
%%sql -r res_tsk_orders
-- Task enfant : transformation des commandes
CREATE OR REPLACE TASK SHOPFLOW_DB.RAW.TSK_LOAD_STG_ORDERS
  WAREHOUSE = WH_TRANSFORM
  COMMENT = 'Charge les nouvelles commandes nettoyées dans STG_ORDERS'
  AFTER SHOPFLOW_DB.RAW.TSK_ROOT_SCHEDULER
  WHEN SYSTEM$STREAM_HAS_DATA('SHOPFLOW_DB.RAW.STR_ORDERS')
AS
  INSERT INTO SHOPFLOW_DB.STAGING.STG_ORDERS (ORDER_ID, CUSTOMER_ID, ORDER_DATE, STATUS, TOTAL_AMOUNT)
  SELECT
    ORDER_ID,
    CUSTOMER_ID,
    TRY_TO_DATE(ORDER_DATE) AS ORDER_DATE,
    UPPER(TRIM(STATUS)) AS STATUS,
    TRY_TO_NUMBER(TOTAL_AMOUNT, 12, 2) AS TOTAL_AMOUNT
  FROM SHOPFLOW_DB.RAW.STR_ORDERS;

In [ ]:
%%sql -r res_tsk_items
-- Task enfant : transformation des items de commande
CREATE OR REPLACE TASK SHOPFLOW_DB.RAW.TSK_LOAD_STG_ORDER_ITEMS
  WAREHOUSE = WH_TRANSFORM
  COMMENT = 'Charge les nouveaux items nettoyés dans STG_ORDER_ITEMS'

  AFTER SHOPFLOW_DB.RAW.TSK_ROOT_SCHEDULER
  WHEN SYSTEM$STREAM_HAS_DATA('SHOPFLOW_DB.RAW.STR_ORDER_ITEMS')
  AS
  INSERT INTO SHOPFLOW_DB.STAGING.STG_ORDER_ITEMS (ORDER_ITEM_ID, ORDER_ID, PRODUCT_ID, QUANTITY, UNIT_PRICE, LINE_TOTAL)
  SELECT
    ORDER_ITEM_ID,
    ORDER_ID,
    PRODUCT_ID,
    TRY_TO_NUMBER(QUANTITY, 10, 0) AS QUANTITY,
    TRY_TO_NUMBER(UNIT_PRICE, 10, 2) AS UNIT_PRICE,
    TRY_TO_NUMBER(QUANTITY, 10, 0) * TRY_TO_NUMBER(UNIT_PRICE, 10, 2) AS LINE_TOTAL
  FROM SHOPFLOW_DB.RAW.STR_ORDER_ITEMS;

In [ ]:
%%sql -r res_tsk_events
-- Task enfant : transformation des événements web
CREATE OR REPLACE TASK SHOPFLOW_DB.RAW.TSK_LOAD_STG_WEB_EVENTS
  WAREHOUSE = WH_TRANSFORM
    COMMENT = 'Charge les nouveaux événements web nettoyés dans STG_WEB_EVENTS'
  AFTER SHOPFLOW_DB.RAW.TSK_ROOT_SCHEDULER
  WHEN SYSTEM$STREAM_HAS_DATA('SHOPFLOW_DB.RAW.STR_WEB_EVENTS')
AS
  INSERT INTO SHOPFLOW_DB.STAGING.STG_WEB_EVENTS (EVENT_ID, EVENT_TYPE, USER_ID, SESSION_ID, EVENT_TS, DEVICE, PRODUCT_ID, IP_ADDRESS, REFERRER, USER_AGENT)
  SELECT
    e.EVENT_ID,
    e.EVENT_TYPE,
    e.USER_ID,
    e.SESSION_ID,
    e.EVENT_TS,
    ext.DEVICE,
    ext.PRODUCT_ID,
    ext.CONTEXT:ip::VARCHAR AS IP_ADDRESS,
    ext.CONTEXT:referrer::VARCHAR AS REFERRER,
    ext.CONTEXT:user_agent::VARCHAR AS USER_AGENT
  FROM SHOPFLOW_DB.RAW.STR_WEB_EVENTS e
  JOIN SHOPFLOW_DB.RAW.WEB_EVENTS_EXT ext ON e.EVENT_ID = ext.EVENT_ID;

In [ ]:
%%sql -r res_tsk_products
-- Task enfant : transformation des produits (VARIANT → colonnes)
CREATE OR REPLACE TASK SHOPFLOW_DB.RAW.TSK_LOAD_STG_PRODUCTS
  WAREHOUSE = WH_TRANSFORM
    COMMENT = 'Charge les nouveaux produits parsés dans STG_PRODUCTS'

  AFTER SHOPFLOW_DB.RAW.TSK_ROOT_SCHEDULER
  WHEN SYSTEM$STREAM_HAS_DATA('SHOPFLOW_DB.RAW.STR_PRODUCTS')
AS
  INSERT INTO SHOPFLOW_DB.STAGING.STG_PRODUCTS (PRODUCT_ID, NAME, BRAND, CATEGORY, PRICE, COLOR, WARRANTY_MONTHS, WEIGHT_G, TAGS)
  SELECT
    DATA:product_id::VARCHAR,
    DATA:name::VARCHAR,
    DATA:brand::VARCHAR,
    DATA:category::VARCHAR,
    DATA:price::NUMBER(10,2),
    DATA:attributes.color::VARCHAR,
    DATA:attributes.warranty_months::INT,
    DATA:attributes.weight_g::INT,
    DATA:tags
  FROM SHOPFLOW_DB.RAW.STR_PRODUCTS;

In [ ]:
%%sql -r res_tsk_customers
-- Task enfant : transformation des clients
CREATE OR REPLACE TASK SHOPFLOW_DB.RAW.TSK_LOAD_STG_CUSTOMERS
  WAREHOUSE = WH_TRANSFORM
  COMMENT = 'Charge les nouveaux clients nettoyés dans STG_CUSTOMERS'

  AFTER SHOPFLOW_DB.RAW.TSK_ROOT_SCHEDULER
  WHEN SYSTEM$STREAM_HAS_DATA('SHOPFLOW_DB.RAW.STR_CUSTOMERS')
AS
  INSERT INTO SHOPFLOW_DB.STAGING.STG_CUSTOMERS (CUSTOMER_ID, FIRST_NAME, LAST_NAME, FULL_NAME, EMAIL, CITY, COUNTRY, CREATED_AT)
  SELECT
    CUSTOMER_ID,
    INITCAP(TRIM(FIRST_NAME)),
    INITCAP(TRIM(LAST_NAME)),
    INITCAP(TRIM(FIRST_NAME)) || ' ' || INITCAP(TRIM(LAST_NAME)),
    LOWER(TRIM(EMAIL)),
    TRIM(CITY),
    UPPER(TRIM(COUNTRY)),
    TRY_TO_TIMESTAMP_NTZ(CREATED_AT)
  FROM SHOPFLOW_DB.RAW.STR_CUSTOMERS;

## 5. Activation des Tasks

**Important :** les Tasks sont créées à l'état `SUSPENDED` par défaut. Il faut les activer (RESUME) pour qu'elles s'exécutent.

**Ordre d'activation :** on doit d'abord activer les tasks enfants, puis la task root. Sinon, la root pourrait se déclencher avant que les enfants soient prêts.

In [ ]:
%%sql -r res_resume_tasks
-- Activer les tasks enfants d'abord
ALTER TASK SHOPFLOW_DB.RAW.TSK_LOAD_STG_ORDERS RESUME;
ALTER TASK SHOPFLOW_DB.RAW.TSK_LOAD_STG_ORDER_ITEMS RESUME;
ALTER TASK SHOPFLOW_DB.RAW.TSK_LOAD_STG_WEB_EVENTS RESUME;
ALTER TASK SHOPFLOW_DB.RAW.TSK_LOAD_STG_PRODUCTS RESUME;
ALTER TASK SHOPFLOW_DB.RAW.TSK_LOAD_STG_CUSTOMERS RESUME;

-- Puis activer la task root
ALTER TASK SHOPFLOW_DB.RAW.TSK_ROOT_SCHEDULER RESUME;

In [ ]:
%%sql -r res_show_tasks
-- Vérifier l'état des tasks
SHOW TASKS IN SCHEMA SHOPFLOW_DB.RAW;

## 6. Chargement initial (consommer les données déjà en RAW)

Les Streams contiennent déjà les données existantes dans RAW (si elles ont été insérées après la création du Stream). Pour peupler STAGING immédiatement sans attendre la prochaine exécution planifiée, on peut déclencher manuellement le DAG de Tasks.

In [ ]:
%%sql -r res_exec_root
-- Exécution manuelle immédiate du DAG
EXECUTE TASK SHOPFLOW_DB.RAW.TSK_ROOT_SCHEDULER;

In [ ]:
%%sql -r res_stg_counts
-- Vérifier les compteurs STAGING après exécution (attendre ~30s que les tasks enfants s'exécutent)
SELECT 'STG_ORDERS' AS table_name, COUNT(*) AS row_count FROM SHOPFLOW_DB.STAGING.STG_ORDERS
UNION ALL SELECT 'STG_ORDER_ITEMS', COUNT(*) FROM SHOPFLOW_DB.STAGING.STG_ORDER_ITEMS
UNION ALL SELECT 'STG_WEB_EVENTS', COUNT(*) FROM SHOPFLOW_DB.STAGING.STG_WEB_EVENTS
UNION ALL SELECT 'STG_PRODUCTS', COUNT(*) FROM SHOPFLOW_DB.STAGING.STG_PRODUCTS
UNION ALL SELECT 'STG_CUSTOMERS', COUNT(*) FROM SHOPFLOW_DB.STAGING.STG_CUSTOMERS;

## 7. Test dynamique — Lot J2

**Scénario de test :**
1. Uploader les fichiers du lot J2 (`orders_j2.csv`, `web_events_j2.json`) dans le stage
2. Les charger dans les tables RAW via COPY INTO
3. Observer que les Streams se remplissent avec les nouveaux enregistrements
4. Attendre la prochaine exécution de la Task (ou la déclencher manuellement)
5. Vérifier que STAGING contient les nouvelles données

Ce test prouve que le pipeline est **incrémental** : seules les nouvelles données sont transformées.

In [ ]:
%%sql -r res_load_j2
-- Étape 1 : Charger le lot J2 dans RAW (après avoir uploadé les fichiers dans le stage)
-- Décommenter après avoir uploadé orders_j2.csv et web_events_j2.json dans le stage

 COPY INTO SHOPFLOW_DB.RAW.RAW_ORDERS
 FROM @SHOPFLOW_DB.RAW.STAGE_LANDING/orders_j2
 FILE_FORMAT = (FORMAT_NAME = 'SHOPFLOW_DB.RAW.FF_CSV_ORDERS')
 ON_ERROR = 'CONTINUE';

-- Pour web_events_j2.json, charger dans la table externe ou directement :
 COPY INTO SHOPFLOW_DB.RAW.WEB_EVENTS_RAW (EVENT_ID, EVENT_TYPE, USER_ID, PAGE_URL, EVENT_TS, SESSION_ID)
 FROM (
   SELECT
     $1:event_id::VARCHAR,
     $1:event_type::VARCHAR,
     $1:user_id::VARCHAR,
     NULL,
     $1:event_ts::TIMESTAMP_NTZ,
     $1:session_id::VARCHAR
   FROM @SHOPFLOW_DB.RAW.STAGE_LANDING/web_events_j2
 )
 FILE_FORMAT = (TYPE = 'JSON')
 ON_ERROR = 'CONTINUE';

SELECT 'Instructions J2 prêtes - décommenter après upload' AS status;

In [ ]:
%%sql -r res_stream_pending
-- Étape 2 : Vérifier que les streams ont capté les nouvelles données
SELECT 'STR_ORDERS' AS stream_name, COUNT(*) AS pending_rows FROM SHOPFLOW_DB.RAW.STR_ORDERS
UNION ALL SELECT 'STR_ORDER_ITEMS', COUNT(*) FROM SHOPFLOW_DB.RAW.STR_ORDER_ITEMS
UNION ALL SELECT 'STR_WEB_EVENTS', COUNT(*) FROM SHOPFLOW_DB.RAW.STR_WEB_EVENTS
UNION ALL SELECT 'STR_PRODUCTS', COUNT(*) FROM SHOPFLOW_DB.RAW.STR_PRODUCTS
UNION ALL SELECT 'STR_CUSTOMERS', COUNT(*) FROM SHOPFLOW_DB.RAW.STR_CUSTOMERS;

In [ ]:
%%sql -r res_exec_j2
-- Étape 3 : Déclencher manuellement le pipeline (ou attendre 5 min)
EXECUTE TASK SHOPFLOW_DB.RAW.TSK_ROOT_SCHEDULER;

In [ ]:
%%sql -r res_task_history
-- Étape 4 : Historique d'exécution des tasks (vérifier succès)
SELECT
  NAME,
  STATE,
  SCHEDULED_TIME,
  COMPLETED_TIME,
  ERROR_CODE,
  ERROR_MESSAGE
FROM TABLE(SHOPFLOW_DB.INFORMATION_SCHEMA.TASK_HISTORY(
  SCHEDULED_TIME_RANGE_START => DATEADD(HOUR, -1, CURRENT_TIMESTAMP()),
  RESULT_LIMIT => 20
))
WHERE SCHEMA_NAME = 'RAW'
ORDER BY SCHEDULED_TIME DESC;

In [ ]:
%%sql -r res_final_counts
-- Étape 5 : Vérifier les compteurs finaux STAGING
SELECT 'STG_ORDERS' AS table_name, COUNT(*) AS rows_count FROM SHOPFLOW_DB.STAGING.STG_ORDERS
UNION ALL SELECT 'STG_ORDER_ITEMS', COUNT(*) FROM SHOPFLOW_DB.STAGING.STG_ORDER_ITEMS
UNION ALL SELECT 'STG_WEB_EVENTS', COUNT(*) FROM SHOPFLOW_DB.STAGING.STG_WEB_EVENTS
UNION ALL SELECT 'STG_PRODUCTS', COUNT(*) FROM SHOPFLOW_DB.STAGING.STG_PRODUCTS
UNION ALL SELECT 'STG_CUSTOMERS', COUNT(*) FROM SHOPFLOW_DB.STAGING.STG_CUSTOMERS;

## Résumé du pipeline

| Composant | Rôle |
|-----------|------|
| **Streams** (STR_*) | Détectent les INSERT dans RAW (CDC append-only) |
| **Tables STAGING** (STG_*) | Stockent les données nettoyées et typées |
| **Task Root** (TSK_ROOT_SCHEDULER) | Déclencheur CRON toutes les 5 min |
| **Tasks enfants** (TSK_LOAD_STG_*) | Transforment les données si le Stream a du contenu |
| **WHEN clause** | `SYSTEM$STREAM_HAS_DATA()` évite les exécutions à vide |

**Flux complet :**
```
Fichiers → Stage → COPY INTO → RAW → Stream détecte → Task transforme → STAGING
```